In [104]:
import pandas as pd
import numpy as np
from IPython.display import display, HTML

def check_my_score(df):
    """
    Evaluates the student's DataFrame against the Data Preprocessing marking scheme.
    """
    score = 0
    feedback = []

    try:
        # Create a copy to prevent accidental modification
        test_df = df.copy()

        # --- INPUT SAFETY ---
        if len(test_df) == 0:
            print("❌ Invalid Data: DataFrame is empty. Please reload the dataset.")
            return

        # Ensure Employee_ID is accessible (in case they set it as index)
        if 'Employee_ID' not in test_df.columns and test_df.index.name == 'Employee_ID':
            test_df = test_df.reset_index()
        elif 'Employee_ID' not in test_df.columns:
            print("❌ Invalid Data: 'Employee_ID' column is missing. Did you drop it?")
            return

        def get_row(emp_id):
            row = test_df[test_df['Employee_ID'] == emp_id]
            return row.iloc[0] if not row.empty else None

        # --- 1. OUTLIER REMOVAL (20 Pts) ---
        row7 = get_row(7)
        if row7 is None:
            score += 10
            feedback.append("✅ Hours Outlier removed")
        elif row7.get('Weekly_Hours', 0) > 20:
            score += 10
            feedback.append("✅ Hours Outlier capped")
        else:
            feedback.append("❌ Low Outlier (5 hrs) still exists")

        row2 = get_row(2)
        if row2 is None:
            score += 10
            feedback.append("✅ Perf Outlier removed")
        elif row2.get('Performance_Score', 0) < 2000:
            score += 10
            feedback.append("✅ Perf Outlier capped")
        else:
            feedback.append("❌ Perf Outlier (9000) still exists")


        # --- 2. SCALING CHECK (20 Pts) ---
        is_scaled = False
        if 'Salary_LPA' in test_df.columns:
            # Filter numeric values only just in case
            sal_vals = pd.to_numeric(test_df['Salary_LPA'], errors='coerce').dropna()
            if len(sal_vals) > 0:
                max_val, min_val = sal_vals.max(), sal_vals.min()
                if 0.9 < max_val <= 1.01 and min_val >= -0.01:
                    is_scaled = True
                    score += 20
                    feedback.append("✅ Data Scaled (MinMax)")
                elif max_val > 1.5:
                    feedback.append("❌ Data not Scaled yet")

        # --- 3. IMPUTATION CHECK (30 Pts) ---
        # A. Age Check (Row 5)
        row5 = get_row(5)
        if row5 is None:
            feedback.append("⛔ WRONG: Row 5 Dropped (Should Fill Age)")
        elif pd.isna(row5.get('Age')):
            feedback.append("❌ Row 5 Age Empty")
        else:
            val = float(row5['Age'])
            if is_scaled:
                if 0.30 <= val <= 0.45:
                    score += 10; feedback.append("✅ Age Filled (Scaled)")
                else:
                    feedback.append("⚠️ Age Value Incorrect")
            else:
                if 26.9 <= val <= 27.1:
                    score += 10; feedback.append("✅ Age Median Correct")
                elif 27.6 <= val <= 27.9:
                    score += 5; feedback.append("⚠️ Used Mean not Median for Age")
                else:
                    feedback.append("⚠️ Age Value Incorrect")

        # B. Salary Check (Row 8)
        row8 = get_row(8)
        if row8 is None:
            feedback.append("⛔ WRONG: Row 8 Dropped (Should Fill Salary)")
        elif pd.isna(row8.get('Salary_LPA')):
            feedback.append("❌ Row 8 Salary Empty")
        else:
            val = float(row8['Salary_LPA'])
            if is_scaled:
                if 0.25 <= val <= 0.35:
                    score += 10; feedback.append("✅ Salary Filled (Scaled)")
                else:
                    feedback.append("⚠️ Salary Value Incorrect")
            else:
                if 14.4 <= val <= 14.6:
                    score += 10; feedback.append("✅ Salary Median Correct")
                else:
                    score += 5; feedback.append("⚠️ Salary (Mean/Other used)")

        # C. Performance Check (Row 4)
        row4 = get_row(4)
        if row4 is None:
            feedback.append("⛔ WRONG: Row 4 Dropped (Should Fill Perf)")
        elif pd.isna(row4.get('Performance_Score')):
            feedback.append("❌ Row 4 Perf Empty")
        else:
            val = float(row4['Performance_Score'])
            if is_scaled:
                if 0.42 <= val <= 0.53:
                    score += 10; feedback.append("✅ Perf Filled (Scaled)")
                elif val < 0.1:
                    feedback.append("❌ Scaled with Outlier present!")
                else:
                    score += 5; feedback.append("⚠️ Perf Value Deviation")
            else:
                if 775 <= val <= 785:
                    score += 10; feedback.append("✅ Perf Median Correct")
                else:
                    score += 5; feedback.append("⚠️ Perf Filled (Mean/Order issue)")

        # --- 4. ENCODING CHECK (30 Pts) ---
        # Education
        if 'Education' in test_df.columns:
            if pd.api.types.is_numeric_dtype(test_df['Education']):
                score += 15
                feedback.append("✅ Label Encoded (Education)")
            else:
                feedback.append("❌ Education is Text")
        else:
            feedback.append("❌ Education Col Missing")

        # Gender
        has_gender = 'Gender' in test_df.columns
        encoded_cols = [c for c in test_df.columns if 'Gender_' in c or str(c).lower() in ['male', 'female']]

        if not has_gender and len(encoded_cols) >= 2:
            score += 15
            feedback.append("✅ One-Hot Encoded (Gender)")
        elif not has_gender and len(encoded_cols) == 1:
            score += 10
            feedback.append("⚠️ One-Hot (drop_first used)")
        elif has_gender and len(encoded_cols) >= 1:
            score += 5
            feedback.append("⚠️ Gender Col kept")
        else:
            feedback.append("❌ One-Hot Pending")

        # --- OUTPUT DISPLAY ---
        final_score = min(round(score), 100)
        color = "green" if final_score == 100 else ("orange" if final_score > 50 else "red")

        html_output = f"""
        <div style="border: 2px solid {color}; border-radius: 10px; padding: 15px; background-color: #f9f9f9; font-family: Arial;">
            <h2 style="color: {color}; margin-top: 0;">🏆 Current Score: {final_score}%</h2>
            <h4 style="margin-bottom: 5px;">Agent Feedback:</h4>
            <ul style="list-style-type: none; padding-left: 0; line-height: 1.6;">
        """
        for item in feedback:
            html_output += f"<li>{item}</li>"

        html_output += "</ul></div>"
        display(HTML(html_output))

    except Exception as e:
        print(f"❌ CRITICAL ERROR IN GRADER: {str(e)}")
        print("Please reload your dataset and try again.")

In [105]:
import pandas as pd
import numpy as np

In [106]:
df = pd.read_csv('Data_Face_Off.csv')
df.head()

,Employee_ID,Gender,Education,City,Age,Salary_LPA,Performance_Score,Weekly_Hours
0,1,Male,Bachelor,Bangalore,24.0,12.5,750.0,45
1,2,Female,Master,Mumbai,28.0,18.0,9000.0,42
2,3,Male,PhD,Delhi,35.0,24.0,820.0,40
3,4,Female,Bachelor,Bangalore,23.0,11.0,NaN,48
4,5,Male,Bachelor,Mumbai,NaN,12.0,690.0,45


In [107]:
df.isnull().sum()

,0
Employee_ID,0
Gender,0
Education,0
City,0
Age,1
Salary_LPA,3
Performance_Score,4
Weekly_Hours,0


In [108]:
df['Age'] = df['Age'].fillna(df['Age'].median())

In [109]:

df['Salary_LPA'] = df['Salary_LPA'].fillna(df['Salary_LPA'].median())

In [110]:
df["Performance_Score"].median()

795.0

In [111]:
df["Performance_Score"] = df["Performance_Score"].fillna(df['Performance_Score'].median())

In [112]:
df["Performance_Score"].median()

795.0

In [113]:
df.isnull().sum()

,0
Employee_ID,0
Gender,0
Education,0
City,0
Age,0
Salary_LPA,0
Performance_Score,0
Weekly_Hours,0


In [114]:
check_my_score(df)

In [115]:
q1 = df['Age'].quantile(0.25)
q3 = df['Age'].quantile(0.75)

iqr = q3 - q1

lower_limit = q1 - 1.5 * iqr
upper_limit = q3 + 1.5 * iqr

df = df[(df['Age'] >= lower_limit) & (df['Age'] <= upper_limit)]
df

,Employee_ID,Gender,Education,City,Age,Salary_LPA,Performance_Score,Weekly_Hours
0,1,Male,Bachelor,Bangalore,24.0,12.5,750.0,45
1,2,Female,Master,Mumbai,28.0,18.0,9000.0,42
2,3,Male,PhD,Delhi,35.0,24.0,820.0,40
3,4,Female,Bachelor,Bangalore,23.0,11.0,795.0,48
4,5,Male,Bachelor,Mumbai,27.0,12.0,690.0,45
5,6,Female,Master,Delhi,29.0,19.5,880.0,43
6,7,Male,Bachelor,Bangalore,25.0,13.0,710.0,5
7,8,Female,PhD,Mumbai,32.0,14.5,850.0,41
8,9,Male,Master,Delhi,27.0,16.5,780.0,44
9,10,Female,Bachelor,Bangalore,24.0,12.0,795.0,46


In [116]:
q1 = df['Weekly_Hours'].quantile(0.25)
q3 = df['Weekly_Hours'].quantile(0.75)

iqr = q3 - q1

lower_limit = q1 - 1.5 * iqr
upper_limit = q3 + 1.5 * iqr

df = df[(df['Weekly_Hours'] >= lower_limit) & (df['Weekly_Hours'] <= upper_limit)]
df


,Employee_ID,Gender,Education,City,Age,Salary_LPA,Performance_Score,Weekly_Hours
0,1,Male,Bachelor,Bangalore,24.0,12.5,750.0,45
1,2,Female,Master,Mumbai,28.0,18.0,9000.0,42
2,3,Male,PhD,Delhi,35.0,24.0,820.0,40
3,4,Female,Bachelor,Bangalore,23.0,11.0,795.0,48
4,5,Male,Bachelor,Mumbai,27.0,12.0,690.0,45
5,6,Female,Master,Delhi,29.0,19.5,880.0,43
7,8,Female,PhD,Mumbai,32.0,14.5,850.0,41
8,9,Male,Master,Delhi,27.0,16.5,780.0,44
9,10,Female,Bachelor,Bangalore,24.0,12.0,795.0,46
10,11,Male,Bachelor,Mumbai,26.0,14.0,720.0,45


In [117]:
q1 = df['Performance_Score'].quantile(0.25)
q3 = df['Performance_Score'].quantile(0.75)

iqr = q3 - q1

lower_limit = q1 - 1.5 * iqr
upper_limit = q3 + 1.5 * iqr

df = df[(df['Performance_Score'] >= lower_limit) & (df['Performance_Score'] <= upper_limit)]
df

,Employee_ID,Gender,Education,City,Age,Salary_LPA,Performance_Score,Weekly_Hours
0,1,Male,Bachelor,Bangalore,24.0,12.5,750.0,45
2,3,Male,PhD,Delhi,35.0,24.0,820.0,40
3,4,Female,Bachelor,Bangalore,23.0,11.0,795.0,48
4,5,Male,Bachelor,Mumbai,27.0,12.0,690.0,45
5,6,Female,Master,Delhi,29.0,19.5,880.0,43
7,8,Female,PhD,Mumbai,32.0,14.5,850.0,41
8,9,Male,Master,Delhi,27.0,16.5,780.0,44
9,10,Female,Bachelor,Bangalore,24.0,12.0,795.0,46
10,11,Male,Bachelor,Mumbai,26.0,14.0,720.0,45
11,12,Female,Master,Delhi,30.0,14.5,890.0,42


In [118]:
print(lower_limit)
print(upper_limit)

627.5
967.5


In [119]:
df

,Employee_ID,Gender,Education,City,Age,Salary_LPA,Performance_Score,Weekly_Hours
0,1,Male,Bachelor,Bangalore,24.0,12.5,750.0,45
2,3,Male,PhD,Delhi,35.0,24.0,820.0,40
3,4,Female,Bachelor,Bangalore,23.0,11.0,795.0,48
4,5,Male,Bachelor,Mumbai,27.0,12.0,690.0,45
5,6,Female,Master,Delhi,29.0,19.5,880.0,43
7,8,Female,PhD,Mumbai,32.0,14.5,850.0,41
8,9,Male,Master,Delhi,27.0,16.5,780.0,44
9,10,Female,Bachelor,Bangalore,24.0,12.0,795.0,46
10,11,Male,Bachelor,Mumbai,26.0,14.0,720.0,45
11,12,Female,Master,Delhi,30.0,14.5,890.0,42


In [120]:
check_my_score(df)

In [121]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

df['Education'] = le.fit_transform(df['Education'])
df.head()

/tmp/ipykernel_1099/2524309021.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Education'] = le.fit_transform(df['Education'])


,Employee_ID,Gender,Education,City,Age,Salary_LPA,Performance_Score,Weekly_Hours
0,1,Male,0,Bangalore,24.0,12.5,750.0,45
2,3,Male,2,Delhi,35.0,24.0,820.0,40
3,4,Female,0,Bangalore,23.0,11.0,795.0,48
4,5,Male,0,Mumbai,27.0,12.0,690.0,45
5,6,Female,1,Delhi,29.0,19.5,880.0,43


In [122]:
df = pd.get_dummies(df, columns=['Gender'])
df.head()

,Employee_ID,Education,City,Age,Salary_LPA,Performance_Score,Weekly_Hours,Gender_Female,Gender_Male
0,1,0,Bangalore,24.0,12.5,750.0,45,False,True
2,3,2,Delhi,35.0,24.0,820.0,40,False,True
3,4,0,Bangalore,23.0,11.0,795.0,48,True,False
4,5,0,Mumbai,27.0,12.0,690.0,45,False,True
5,6,1,Delhi,29.0,19.5,880.0,43,True,False


In [123]:
df = pd.get_dummies(df, columns=['City'])
df.head()

,Employee_ID,Education,Age,Salary_LPA,Performance_Score,Weekly_Hours,Gender_Female,Gender_Male,City_Bangalore,City_Delhi,City_Mumbai
0,1,0,24.0,12.5,750.0,45,False,True,True,False,False
2,3,2,35.0,24.0,820.0,40,False,True,False,True,False
3,4,0,23.0,11.0,795.0,48,True,False,True,False,False
4,5,0,27.0,12.0,690.0,45,False,True,False,False,True
5,6,1,29.0,19.5,880.0,43,True,False,False,True,False


In [124]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

df[["Age", "Salary_LPA", "Performance_Score"]] = scaler.fit_transform(
    df[["Age", "Salary_LPA", "Performance_Score"]]
)

In [125]:
df

,Employee_ID,Education,Age,Salary_LPA,Performance_Score,Weekly_Hours,Gender_Female,Gender_Male,City_Bangalore,City_Delhi,City_Mumbai
0,1,0,0.153846,0.148148,0.333333,45,False,True,True,False,False
2,3,2,1.000000,1.000000,0.666667,40,False,True,False,True,False
3,4,0,0.076923,0.037037,0.547619,48,True,False,True,False,False
4,5,0,0.384615,0.111111,0.047619,45,False,True,False,False,True
5,6,1,0.538462,0.666667,0.952381,43,True,False,False,True,False
7,8,2,0.769231,0.296296,0.809524,41,True,False,False,False,True
8,9,1,0.384615,0.444444,0.476190,44,False,True,False,True,False
9,10,0,0.153846,0.111111,0.547619,46,True,False,True,False,False
10,11,0,0.307692,0.259259,0.190476,45,False,True,False,False,True
11,12,1,0.615385,0.296296,1.000000,42,True,False,False,True,False


In [126]:
check_my_score(df)